In [1]:
import pandas as pd
import numpy as np
import random

np.random.seed(42)

# ------------------------
# Matches info
# ------------------------
matches_df = pd.DataFrame({
    "match_id": [1,2,3,4,5],
    "season": [2021,2021,2022,2022,2023],
    "date": [
        "2021-04-01",
        "2021-04-05",
        "2022-04-02",
        "2022-04-06",
        "2023-04-03"
    ],
    "venue": [
        "Wankhede Stadium",
        "Eden Gardens",
        "Chinnaswamy Stadium",
        "Arun Jaitley Stadium",
        "Wankhede Stadium"
    ],
    "team1": [
        "Mumbai Warriors",
        "Delhi Kings",
        "Chennai Titans",
        "Punjab Lions",
        "Kolkata Knights"
    ],
    "team2": [
        "Delhi Kings",
        "Mumbai Warriors",
        "Punjab Lions",
        "Chennai Titans",
        "Bangalore Rockets"
    ],
    "winner": [
        "Mumbai Warriors",
        "Delhi Kings",
        "Punjab Lions",
        "Chennai Titans",
        "Kolkata Knights"
    ]
})

# ------------------------
# Players setup
# ------------------------
batters = [f"Player_{i}" for i in range(1,21)]
bowlers = [f"Bowler_{i}" for i in range(1,11)]

# ------------------------
# Generate deliveries
# ------------------------
deliveries = []

for _, match in matches_df.iterrows():
    for inning in [1,2]:
        batting_team = match['team1'] if inning==1 else match['team2']
        bowling_team = match['team2'] if inning==1 else match['team1']
        batting_order = random.sample(batters, 11)
        striker_idx = 0
        non_striker_idx = 1
        for over in range(1,21):  # 20 overs
            bowler = random.choice(bowlers)
            for ball in range(1,7):
                batter = batting_order[striker_idx]
                batsman_runs = np.random.choice([0,1,2,3,4,6], p=[0.3,0.3,0.15,0.05,0.15,0.05])
                extra_runs = np.random.choice([0,1], p=[0.95,0.05])
                total_runs = batsman_runs + extra_runs
                is_wicket = np.random.choice([0,1], p=[0.97,0.03])
                dismissal_kind = None
                player_dismissed = None
                if is_wicket:
                    dismissal_kind = random.choice(["bowled","caught","lbw"])
                    player_dismissed = batter
                    striker_idx = min(striker_idx+1, 10)  # next batter
                deliveries.append({
                    "match_id": match['match_id'],
                    "inning": inning,
                    "batting_team": batting_team,
                    "bowling_team": bowling_team,
                    "over": over,
                    "ball": ball,
                    "batter": batter,
                    "bowler": bowler,
                    "batsman_runs": batsman_runs,
                    "extra_runs": extra_runs,
                    "total_runs": total_runs,
                    "is_wicket": is_wicket,
                    "dismissal_kind": dismissal_kind,
                    "player_dismissed": player_dismissed
                })
                # Swap strike if runs odd
                if batsman_runs % 2 == 1:
                    striker_idx, non_striker_idx = non_striker_idx, striker_idx
            # Swap strike at end of over
            striker_idx, non_striker_idx = non_striker_idx, striker_idx

deliveries_df = pd.DataFrame(deliveries)

# ------------------------
# Quick look
# ------------------------
print(deliveries_df.shape)
deliveries_df.head(15)

(1200, 14)


,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,batsman_runs,extra_runs,total_runs,is_wicket,dismissal_kind,player_dismissed
0,1,1,Mumbai Warriors,Delhi Kings,1,1,Player_13,Bowler_10,1,1,2,0,NaN,NaN
1,1,1,Mumbai Warriors,Delhi Kings,1,2,Player_9,Bowler_10,1,0,1,0,NaN,NaN
2,1,1,Mumbai Warriors,Delhi Kings,1,3,Player_13,Bowler_10,0,0,0,0,NaN,NaN
3,1,1,Mumbai Warriors,Delhi Kings,1,4,Player_13,Bowler_10,2,0,2,0,NaN,NaN
4,1,1,Mumbai Warriors,Delhi Kings,1,5,Player_13,Bowler_10,4,0,4,0,NaN,NaN
5,1,1,Mumbai Warriors,Delhi Kings,1,6,Player_13,Bowler_10,0,0,0,0,NaN,NaN
6,1,1,Mumbai Warriors,Delhi Kings,2,1,Player_9,Bowler_10,1,0,1,0,NaN,NaN
7,1,1,Mumbai Warriors,Delhi Kings,2,2,Player_13,Bowler_10,0,0,0,0,NaN,NaN
8,1,1,Mumbai Warriors,Delhi Kings,2,3,Player_13,Bowler_10,1,0,1,0,NaN,NaN
9,1,1,Mumbai Warriors,Delhi Kings,2,4,Player_9,Bowler_10,1,0,1,0,NaN,NaN


In [2]:
'''
Build master table
'''
df = deliveries_df.merge(matches_df,on="match_id",how="left")
df

,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,batsman_runs,extra_runs,total_runs,is_wicket,dismissal_kind,player_dismissed,season,date,venue,team1,team2,winner
0,1,1,Mumbai Warriors,Delhi Kings,1,1,Player_13,Bowler_10,1,1,2,0,NaN,NaN,2021,2021-04-01,Wankhede Stadium,Mumbai Warriors,Delhi Kings,Mumbai Warriors
1,1,1,Mumbai Warriors,Delhi Kings,1,2,Player_9,Bowler_10,1,0,1,0,NaN,NaN,2021,2021-04-01,Wankhede Stadium,Mumbai Warriors,Delhi Kings,Mumbai Warriors
2,1,1,Mumbai Warriors,Delhi Kings,1,3,Player_13,Bowler_10,0,0,0,0,NaN,NaN,2021,2021-04-01,Wankhede Stadium,Mumbai Warriors,Delhi Kings,Mumbai Warriors
3,1,1,Mumbai Warriors,Delhi Kings,1,4,Player_13,Bowler_10,2,0,2,0,NaN,NaN,2021,2021-04-01,Wankhede Stadium,Mumbai Warriors,Delhi Kings,Mumbai Warriors
4,1,1,Mumbai Warriors,Delhi Kings,1,5,Player_13,Bowler_10,4,0,4,0,NaN,NaN,2021,2021-04-01,Wankhede Stadium,Mumbai Warriors,Delhi Kings,Mumbai Warriors
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,5,2,Bangalore Rockets,Kolkata Knights,20,2,Player_16,Bowler_4,1,0,1,0,NaN,NaN,2023,2023-04-03,Wankhede Stadium,Kolkata Knights,Bangalore Rockets,Kolkata Knights
1196,5,2,Bangalore Rockets,Kolkata Knights,20,3,Player_18,Bowler_4,0,0,0,0,NaN,NaN,2023,2023-04-03,Wankhede Stadium,Kolkata Knights,Bangalore Rockets,Kolkata Knights
1197,5,2,Bangalore Rockets,Kolkata Knights,20,4,Player_18,Bowler_4,0,0,0,0,NaN,NaN,2023,2023-04-03,Wankhede Stadium,Kolkata Knights,Bangalore Rockets,Kolkata Knights
1198,5,2,Bangalore Rockets,Kolkata Knights,20,5,Player_18,Bowler_4,1,0,1,0,NaN,NaN,2023,2023-04-03,Wankhede Stadium,Kolkata Knights,Bangalore Rockets,Kolkata Knights


In [3]:
'''
Task 1 - Compute strike rate per batter across games
'''
df_strike_rate_batsmen = (
    df
    .groupby(['batter','match_id'])
    .agg(
        runs=('batsman_runs','sum'),
        balls=('ball','count')
    )
    .assign(batter_strike_rate=lambda x: (x.runs / x.balls) * 100)
)

df_strike_rate_batsmen["rolling_strike_rate"] = (
    df_strike_rate_batsmen
    .groupby("batter")["batter_strike_rate"]
    .transform(lambda x: x.rolling(3, min_periods=1).mean())
)
df_strike_rate_batsmen

runs  balls  batter_strike_rate  rolling_strike_rate
batter    match_id                                                      
Player_1  3           38     22          172.727273           172.727273
          5           57     33          172.727273           172.727273
Player_10 2          156     83          187.951807           187.951807
          3            0      3            0.000000            93.975904
          4           11      5          220.000000           135.983936
Player_12 3           26     17          152.941176           152.941176
          5           51     32          159.375000           156.158088
Player_13 1          104     61          170.491803           170.491803
          2           30     17          176.470588           173.481196
          4          124     71          174.647887           173.870093
          5           55     29          189.655172           180.257883
Player_14 1           66     23          286.956522           286.956522
          3           25     15          166.666667           226.811594
          4            6      2          300.000000           251.207729
          5           10      5          200.000000           222.222222
Player_15 3           27     13          207.692308           207.692308
Player_16 5           22     13          169.230769           169.230769
Player_17 3           30     19          157.894737           157.894737
Player_18 1           13     11          118.181818           118.181818
          3          132     78          169.230769           143.706294
          4           12      9          133.333333           140.248640
          5           25     23          108.695652           137.086585
Player_2  2           55     31          177.419355           177.419355
          3            1      2           50.000000           113.709677
          4           12      8          150.000000           125.806452
Player_20 1           31     24          129.166667           129.166667
Player_3  2           19     12          158.333333           158.333333
          3          107     66          162.121212           160.227273
Player_4  4           87     50          174.000000           174.000000
          5           52     23          226.086957           200.043478
Player_5  2          134     81          165.432099           165.432099
          4           86     54          159.259259           162.345679
Player_6  1          124     64          193.750000           193.750000
          4           35     18          194.444444           194.097222
          5           35     19          184.210526           190.801657
Player_7  2           18     16          112.500000           112.500000
          3           10      5          200.000000           156.250000
Player_8  4           40     23          173.913043           173.913043
Player_9  1           74     57          129.824561           129.824561
          5          109     63          173.015873           151.420217

In [4]:
'''
Task 2 - Best death over bowlers
'''
df_death_over_bowlers = (
    df.loc[lambda o : o['over'] >=16]
    .groupby(['bowler','season'])
    .agg(
        runs_conceeded = ('total_runs','sum'),
        balls_bowled = ('ball','count')
    )
    .assign(
        overs_bowled = lambda x: x.balls_bowled/6,
        economy = lambda x: x.runs_conceeded/x.overs_bowled)
    )
df_death_over_bowlers

runs_conceeded  balls_bowled  overs_bowled    economy
bowler    season                                                       
Bowler_1  2021                39            24           4.0   9.750000
          2022                 9             6           1.0   9.000000
Bowler_10 2021                13             6           1.0  13.000000
          2022                 8             6           1.0   8.000000
Bowler_2  2021                25            18           3.0   8.333333
          2022                32            18           3.0  10.666667
          2023                 7             6           1.0   7.000000
Bowler_3  2021                13             6           1.0  13.000000
          2023                 9             6           1.0   9.000000
Bowler_4  2021                36            24           4.0   9.000000
          2022                 8             6           1.0   8.000000
          2023                16            12           2.0   8.000000
Bowler_5  2021                32            12           2.0  16.000000
          2022                32            12           2.0  16.000000
          2023                19            12           2.0   9.500000
Bowler_6  2021                14             6           1.0  14.000000
          2022                32            24           4.0   8.000000
          2023                19            12           2.0   9.500000
Bowler_7  2022                12             6           1.0  12.000000
          2023                11             6           1.0  11.000000
Bowler_8  2021                13            12           2.0   6.500000
          2022                43            24           4.0  10.750000
Bowler_9  2021                30            12           2.0  15.000000
          2022                43            18           3.0  14.333333
          2023                 8             6           1.0   8.000000